# Sesión 6 · Notebook 1 — Consultar Prometheus desde Python

Grafana sirve para **mirar**. Para automatizar —un informe diario, un chequeo en
CI, una notificación a medida— hace falta código.

Prometheus expone su API en `http://localhost:9090/api/v1/`. Dos endpoints
cubren casi todo:

| Endpoint | Devuelve |
|---|---|
| `GET /api/v1/query` | Un valor **instantáneo**: un punto en el tiempo |
| `GET /api/v1/query_range` | Una **serie temporal**: muchos puntos |

**Antes de empezar:** el stack levantado (`docker compose up -d`) y las
dependencias instaladas (`pip install -r notebooks/requirements.txt`).

> Los nombres de métrica de este notebook salen de `docs/metricas.md`. Todos
> llevan el prefijo `orderflow_`. Si copias una consulta de las diapositivas y te
> devuelve vacío, es casi seguro que le falta ese prefijo.

In [ ]:
import requests

PROM = "http://localhost:9090"


def prom_query(expr):
    """Consulta instantánea. Devuelve la lista de resultados."""
    r = requests.get(f"{PROM}/api/v1/query", params={"query": expr}, timeout=10)
    r.raise_for_status()
    data = r.json()
    assert data["status"] == "success", data
    return data["data"]["result"]


# ¿Cuántas órdenes se han procesado en total?
prom_query("sum(orderflow_orders_processed_total)")

## Comprobar un nombre antes de usarlo

El error más caro de esta sesión es escribir un nombre de métrica que no existe.
Prometheus no protesta: devuelve una lista vacía. La consulta parece correcta y
el resultado parece "es que no hay datos".

Esta función te dice si un nombre existe de verdad.

In [ ]:
def existe(metrica):
    """True si Prometheus conoce esa métrica."""
    r = requests.get(f"{PROM}/api/v1/label/__name__/values", timeout=10)
    return metrica in r.json()["data"]


for nombre in [
    "orderflow_orders_processed_total",   # correcta
    "orders_processed_total",             # sin prefijo: no existe
]:
    print(f"{nombre:<38} {'existe' if existe(nombre) else 'NO EXISTE'}")

## Consulta de rango: una serie temporal

Pedimos la **tasa** de órdenes procesadas por segundo durante la última hora, con
un punto cada 15 segundos.

In [ ]:
import time


def prom_query_range(expr, minutes=60, step="15s"):
    end = time.time()
    start = end - minutes * 60
    r = requests.get(
        f"{PROM}/api/v1/query_range",
        params={"query": expr, "start": start, "end": end, "step": step},
        timeout=15,
    )
    r.raise_for_status()
    return r.json()["data"]["result"]


series = prom_query_range("sum(rate(orderflow_orders_processed_total[1m]))", minutes=60)
print("series devueltas:", len(series))
print(series[0]["values"][:3] if series else "sin datos aún")

## Graficar la tasa de procesamiento

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime

if series:
    valores = series[0]["values"]          # [[timestamp, "valor"], ...]
    xs = [datetime.fromtimestamp(float(t)) for t, _ in valores]
    ys = [float(v) for _, v in valores]

    plt.figure(figsize=(10, 4))
    plt.plot(xs, ys)
    plt.title("Órdenes procesadas por segundo (rate 1m)")
    plt.xlabel("tiempo")
    plt.ylabel("órdenes/s")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No hay datos todavía; deja correr el pipeline unos minutos.")

## Ejercicio

1. Calcula el **percentil 95** de la latencia y grafícalo:

   ```promql
   histogram_quantile(0.95, sum by (le) (rate(orderflow_processing_duration_seconds_bucket[5m])))
   ```

2. Escribe una función `porcentaje_error()` que devuelva el porcentaje de órdenes
   fallidas en los últimos 5 minutos, usando `prom_query`.

   *Pista: la expresión es la misma del panel de la Sesión 4 y de la regla de
   alerta de la Sesión 5. No la reinventes: búscala y reutilízala.*